In [2]:
import numpy as np
import few
import os
import sys
import pickle

dir_work = '/home/svu/e1498138/emri_search/work'
# dir_work = '/nfs/home/svu/e1498138/localgit/FEWNEW/work/'
os.chdir(dir_work)
sys.path.insert(0, dir_work)

from GWfuncs_noise import GravWaveAnalysis, build_waveform_response
from loglike_timemax_noise import LogLike
# from loglike_phasemax_noise import LogLike

import parismc
import cupy as cp

cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

use_gpu = True
tdi_gen = 1
dt = 10
T = 2.5
print(f"Using dt={dt}s, T={T}yr")

print('Building ResponseWrapper...')
tdi_gen_res = 2
waveform_response = build_waveform_response(T=T, dt=dt, use_gpu=True, tdi_gen=tdi_gen_res)
print('Res TDI Gen: ', {tdi_gen_res})
tdi_gen_psd = 1
# print('Building GravWaveAnalysis...')
gwf = GravWaveAnalysis(T=T, dt=dt, use_gpu=use_gpu, tdi_gen=tdi_gen_psd)
print('Res TDI Gen: ', {tdi_gen_psd})

# Source parameters
m1 = 1e6
m2 = 1e1
a = 0.5
p0 = 10.0469014
e0 = 0.1
xI0 = 1.0
dist = 3.87879918  # Gpc
qS = 1.04719755
phiS = 0.785398163
qK =  0.628318531
phiK = 0.523598776
Phi_phi0 = 0.1
Phi_theta0 = 0.2
Phi_r0 = 0.3

params_star = [m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0]
param_true = [np.log10(m1), np.log10(m2), a, p0, e0]

n_vals = np.arange(-1, 6)
ell = 2

print('Initializing LogLike...')
loglike_obj = LogLike(
    params=params_star,
    waveform_response=waveform_response,
    gwf=gwf,
    add_noise=True,
    seed=42,
    verbose=False,
    ell=ell,
    n_vals=n_vals,
    M_mode=None,
)
print('LogLike initialized.')
params_true = [np.log10(m1), np.log10(m2), a, p0, e0]


Using dt=10s, T=2.5yr
Building ResponseWrapper...
Res TDI Gen:  {2}
Res TDI Gen:  {1}
Initializing LogLike...
LogLike initialized.


In [3]:
def log_density(params):
    params = np.asarray(params)
    log_likes = np.zeros(params.shape[0])
    for i in range(params.shape[0]):
        logm1, logm2, a, p0, e0 = params[i]
        try:
            loglike = loglike_obj(np.array([
                10**logm1, 10**logm2, a, p0, e0,
                xI0, dist, qS, phiS, qK, phiK,
                Phi_phi0, Phi_theta0, Phi_r0
            ]))
        except Exception:
            loglike = -np.inf
        log_likes[i] = loglike
    return log_likes

In [4]:
log_density([params_true])

array([27.0461799])